# Preprocessing

In [54]:
import pandas as pd

products = pd.read_csv("/kaggle/input/datasets/sadmanshafiq1611/product-details/product_details.csv")

print(products.head())
print(products.shape)
print(products.info())
print(products.describe())

                                    URL  \
0  https://www.amazon.com/dp/B0GJ5KNC27   
1  https://www.amazon.com/dp/B0FRN2Z4JF   
2  https://www.amazon.com/dp/B09C13PZX7   
3  https://www.amazon.com/dp/B0G12HGHGM   
4  https://www.amazon.com/dp/B0BX52C4LN   

                                               Title       Brand Rating  \
0  GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...   GravaStar    4.5   
1  GravaStar Mercury M1 Pro Wireless Gaming Mouse...   GravaStar    4.5   
2  Razer Basilisk V3 Customizable RGB Wired Ergon...       Razer    4.6   
3  Logitech G PRO X2 SUPERSTRIKE Wireless Gaming ...  Logitech G    4.5   
4  HyperX Pulsefire Haste 2 – Wireless Gaming Mou...      HyperX    4.1   

    Price Review_Count                                          Image_URL  \
0   71.95        2,070  https://m.media-amazon.com/images/I/61M5yEjuIF...   
1   89.96        1,485  https://m.media-amazon.com/images/I/51BhB+GhmQ...   
2   34.98       10,516  https://m.media-amazon.com/images/I

In [55]:
products = products[products["Reviews"] != "not available"]

print(len(products))

1162


In [63]:
import json

products["parsed_reviews"] = products["Reviews"].apply(json.loads)


reviews_df = products.explode("parsed_reviews").reset_index()

normalized = pd.json_normalize(reviews_df["parsed_reviews"])

reviews_df = pd.concat([reviews_df[["URL", "Title", "Brand", "Category"]], normalized], axis=1)
print(reviews_df)

                                        URL  \
0      https://www.amazon.com/dp/B0GJ5KNC27   
1      https://www.amazon.com/dp/B0GJ5KNC27   
2      https://www.amazon.com/dp/B0GJ5KNC27   
3      https://www.amazon.com/dp/B0GJ5KNC27   
4      https://www.amazon.com/dp/B0GJ5KNC27   
...                                     ...   
13098  https://www.amazon.com/dp/B07KF8L7Z5   
13099  https://www.amazon.com/dp/B07KF8L7Z5   
13100  https://www.amazon.com/dp/B07KF8L7Z5   
13101  https://www.amazon.com/dp/B07KF8L7Z5   
13102  https://www.amazon.com/dp/B07KF8L7Z5   

                                                   Title      Brand  \
0      GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...  GravaStar   
1      GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...  GravaStar   
2      GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...  GravaStar   
3      GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...  GravaStar   
4      GravaStar Mercury M2 Wireless Gaming Mouse, 5 ...  GravaStar   
...      

In [57]:
!pip install vaderSentiment



In [58]:
print(len(reviews_df))
reviews_df = reviews_df[reviews_df["review_text"].notna()]
print(len(reviews_df))

13103
13095


# Feature Engineering

In [59]:
title_and_text = []

for i, row in reviews_df.iterrows():
    both = str(row["review_title"]) + str(row["review_text"])
    title_and_text.append(both)

reviews_df["title_and_text"] = title_and_text




# Model Selection and Training

In [60]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.metrics import accuracy_score

analyzer = SentimentIntensityAnalyzer()

predictions = []

for i in reviews_df["title_and_text"]:
    score = analyzer.polarity_scores(i)
    compound = score["compound"]
    if compound >= 0.5:
        predictions.append("positive")
    elif compound <= -0.5:
        predictions.append("negative")
    else:
        predictions.append("neutral")

reviews_df["label"] = predictions






    


In [61]:
def summary(column):

    summary_rows = []

    for value in reviews_df[column].unique():
        value_data = reviews_df[reviews_df[column] == value]
        pos = (value_data["label"] == "positive").sum()
        neu = (value_data["label"] == "neutral").sum()
        neg = (value_data["label"] == "negative").sum()
        total = len(value_data)
        pos_percentage = round(100 * pos / total, 1)


        summary_rows.append({
            column: value,
            "Count of Positives": pos,
            "Count of Negatives": neg,
            "Count of Neutrals": neu,
            "Total Reviews": total,
            "Positive Percentage": pos_percentage
            
            
        })
    return pd.DataFrame(summary_rows)


product_summary = summary("URL")


#print(product_summary)

category_summary = summary("Category")

print(category_summary)






        

                                              Category  Count of Positives  \
0               Video Games,PC,Accessories,Gaming Mice                 181   
1    Electronics,Computers & Accessories,Computer A...                  30   
2    Video Games,Legacy Systems,Sega Systems,Sega D...                  11   
3    Electronics,Computers & Accessories,Computer A...                 153   
4    Cell Phones & Accessories,Accessories,Chargers...                  56   
..                                                 ...                 ...   
166  Musical Instruments,Instrument Accessories,Gui...                   8   
167  Musical Instruments,Amplifiers & Effects,Guita...                  10   
168  Musical Instruments,Amplifiers & Effects,Guita...                  10   
169  Musical Instruments,Instrument Accessories,Gui...                   5   
170  Musical Instruments,Instrument Accessories,Gui...                   8   

     Count of Negatives  Count of Neutrals  Total Reviews  Posi

In [62]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


train, test = train_test_split(
    reviews_df, test_size=0.2, random_state=65, stratify=reviews_df["label"]
)

vectorizer = TfidfVectorizer(stop_words="english")
X_train = vectorizer.fit_transform(train["title_and_text"])
X_test = vectorizer.transform(test["title_and_text"])

model = LogisticRegression(random_state=65, max_iter=1000)

model.fit(X_train, train["label"])

predictions = model.predict(X_test)

accuracy = accuracy_score(test["label"], predictions)
print(accuracy)

0.8839251622756777
